# Embed Research Ambit papers with BGE-large (T4 GPU)

This notebook embeds the cleaned paper corpus (`papers_for_embedding.jsonl`) using
**`BAAI/bge-large-en-v1.5`** (1024-dim) on a GPU, and produces a compressed
`paper_embeddings_bge_large.npz` with parallel `ids` and `emb` arrays.

## Before you start
1. **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**, then Save.
2. Have `papers_for_embedding.jsonl` ready (produced locally by `s11_export_for_embedding.py`).
   It is ~85 MB with ~68.6k rows, each line `{"id": ..., "text": ...}`.

## What it does
- Loads the jsonl, encodes documents in batches with `normalize_embeddings=True`,
  saves float16 embeddings, self-checks the shape, and downloads the result.
- Expected runtime on a T4: **~10-20 min** for ~68.6k rows.

## 0. Confirm the GPU
If this shows a Tesla T4 (or similar), you're good. If it errors, set the T4 runtime first.

In [ ]:
!nvidia-smi

## 1. Install dependencies
`sentence-transformers` pulls in `torch` (already on Colab) and `transformers`.

In [ ]:
!pip install -q sentence-transformers

## 2. Provide the input file `papers_for_embedding.jsonl`

Two supported options — pick ONE and run its cell:
- **Option A: direct upload** (simple, good for an 85 MB file).
- **Option B: Google Drive** (survives disconnects; put the file in your Drive first).

After running one option, `INPUT_PATH` will point at the file.

In [ ]:
# ---- Option A: upload from your computer ----
# Uncomment the two lines below to use direct upload.
# from google.colab import files
# _up = files.upload()  # choose papers_for_embedding.jsonl

INPUT_PATH = 'papers_for_embedding.jsonl'  # default if uploaded to the CWD
print('INPUT_PATH =', INPUT_PATH)

In [ ]:
# ---- Option B: mount Google Drive ----
# Uncomment to mount Drive, then set INPUT_PATH to where you placed the file.
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_PATH = '/content/drive/MyDrive/research_ambit/papers_for_embedding.jsonl'
# print('INPUT_PATH =', INPUT_PATH)

In [ ]:
import os, json
assert os.path.exists(INPUT_PATH), f'File not found: {INPUT_PATH} -- run an option cell above.'
size_mb = os.path.getsize(INPUT_PATH) / (1024*1024)
print(f'{INPUT_PATH}: {size_mb:.1f} MB')

## 3. Load the rows
Read `id` and `text` in file order. Order is preserved end-to-end so `ids[i]` matches `emb[i]`.

In [ ]:
ids, texts = [], []
with open(INPUT_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        ids.append(str(rec['id']))
        texts.append(rec['text'])
N = len(ids)
assert len(set(ids)) == N, 'duplicate ids detected in input!'
print(f'loaded {N} rows; example text[:200]:')
print(texts[0][:200])

## 4. A note on BGE prefixes (important)

BGE models support an optional query instruction prefix
(`Represent this sentence for searching relevant passages:`) that is meant **only for the
query side of asymmetric retrieval**. Here we are embedding **documents** and will compare
them to **domain/theme centroids that are themselves documents** (symmetric similarity),
so we embed **plain text with NO prefix** on either side. Keep both sides consistent later.

## 5. Load the model and embed (chunked + resumable)

Embeddings are computed in chunks and each chunk is cached to `emb_parts/`. If the runtime
disconnects, just re-run this cell — completed chunks are skipped. `normalize_embeddings=True`
makes cosine similarity == dot product downstream.

In [ ]:
import numpy as np, torch, time, glob
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

MODEL_NAME = 'BAAI/bge-large-en-v1.5'
BATCH_SIZE = 64            # encode batch size on the GPU
CHUNK_SIZE = 5000         # rows per cached chunk (resume granularity)
PARTS_DIR = 'emb_parts'
os.makedirs(PARTS_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'No GPU! Set Runtime -> Change runtime type -> T4 GPU.'
model = SentenceTransformer(MODEL_NAME, device=device)
print('model loaded on', device)

n_chunks = (N + CHUNK_SIZE - 1) // CHUNK_SIZE
for ci in range(n_chunks):
    part_path = os.path.join(PARTS_DIR, f'part_{ci:04d}.npy')
    if os.path.exists(part_path):
        continue  # already embedded this chunk (resume)
    lo, hi = ci * CHUNK_SIZE, min((ci + 1) * CHUNK_SIZE, N)
    t0 = time.time()
    emb = model.encode(
        texts[lo:hi], batch_size=BATCH_SIZE, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    ).astype(np.float16)
    np.save(part_path, emb)
    print(f'chunk {ci+1}/{n_chunks} rows[{lo}:{hi}] -> {emb.shape} in {time.time()-t0:.0f}s')
print('all chunks done')

## 6. Assemble and save `paper_embeddings_bge_large.npz`
Concatenate the cached chunks in order, pair them with `ids`, and save compressed float16.

In [ ]:
parts = sorted(glob.glob(os.path.join(PARTS_DIR, 'part_*.npy')))
emb_all = np.concatenate([np.load(p) for p in parts], axis=0)
ids_arr = np.array(ids, dtype=object)
assert emb_all.shape[0] == N, (emb_all.shape, N)

OUT_NPZ = 'paper_embeddings_bge_large.npz'
np.savez_compressed(OUT_NPZ, ids=ids_arr, emb=emb_all)
print('saved', OUT_NPZ, f'{os.path.getsize(OUT_NPZ)/(1024*1024):.1f} MB')

## 7. Self-check
Confirms `len(ids) == N` and `emb.shape == (N, 1024)` and reloads cleanly.

In [ ]:
z = np.load(OUT_NPZ, allow_pickle=True)
r_ids, r_emb = z['ids'], z['emb']
print('ids:', r_ids.shape, '| emb:', r_emb.shape, '| dtype:', r_emb.dtype)
assert len(r_ids) == N, (len(r_ids), N)
assert r_emb.shape == (N, 1024), r_emb.shape
assert list(r_ids[:3]) == ids[:3], 'id order mismatch!'
# normalized vectors should have ~unit L2 norm
norms = np.linalg.norm(r_emb[:5].astype(np.float32), axis=1)
print('sample L2 norms (expect ~1.0):', np.round(norms, 4))
print('SELF-CHECK PASSED')

## 8. Download the result
Either download to your computer (Option A) or copy to Drive (Option B).

In [ ]:
# Option A: download to your computer
from google.colab import files
files.download(OUT_NPZ)

In [ ]:
# Option B: copy to Google Drive (requires the Drive mount from step 2)
# import shutil
# dest = '/content/drive/MyDrive/research_ambit/paper_embeddings_bge_large.npz'
# os.makedirs(os.path.dirname(dest), exist_ok=True)
# shutil.copy(OUT_NPZ, dest)
# print('copied to', dest)

## Done

Download **`paper_embeddings_bge_large.npz`** and place it back in your local repo at:

```
classification-pipeline/outputs/work/paper_embeddings_bge_large.npz
```

Then the next pipeline step can load it (verify with `pipeline/s12_verify_embeddings_loader.py`).